### Libs and Functions

In [0]:
import pyspark.sql.functions as F
import requests
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

### Definitions

In [0]:
JSON = "https://health.data.ny.gov/api/views/jxy9-yhdk/rows.json"
NY_HEALTH_TABLE = "health_silver.ny_health_data"

schema = StructType([
    StructField("sid", StringType(), True),
    StructField("id", StringType(), True),
    StructField("position", IntegerType(), True),
    StructField("created_at", IntegerType(), True),
    StructField("created_meta", StringType(), True),
    StructField("updated_at", IntegerType(), True),
    StructField("updated_meta", StringType(), True),
    StructField("meta", StringType(), True),
    StructField("year", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("county", StringType(), True),
    StructField("sex", StringType(), True),
    StructField("count", StringType(), True),
])

### Importing JSON

In [0]:
url = JSON
response = requests.get(url)

try:
    data = response.json()

    colunas = [col["name"] for col in data["meta"]["view"]["columns"]]

except requests.exceptions.RequestException as e:
    print(f"Erro ao fazer a requisição: {e}")

except json.JSONDecodeError as e:
    print(f"Erro ao analisar o JSON: {e}")

In [0]:
df_data = (spark.createDataFrame([data]).select('data'))
df.printSchema()
     

root
 |-- data: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: string (containsNull = true)
 |-- meta: map (nullable = true)
 |    |-- key: string
 |    |-- value: map (valueContainsNull = true)
 |    |    |-- key: string
 |    |    |-- value: string (valueContainsNull = true)



In [0]:
ny_health_data = spark.createDataFrame(data["data"], schema=schema)

### Data Cleaning

In [0]:
ny_health_data = (
    ny_health_data
    .withColumns({
        "created_at": F.from_unixtime(F.col("created_at")).cast("timestamp"),
        "updated_at": F.from_unixtime(F.col("updated_at")).cast("timestamp"),
        "year": F.col("year").cast(IntegerType()),
        "count": F.col("count").cast(IntegerType()),
    })
)

In [0]:
ny_health_data.display()

sid,id,position,created_at,created_meta,updated_at,updated_meta,meta,year,first_name,county,sex,count
row-ddjv_cm93_6icc,00000000-0000-0000-0E58-7483520AF138,0,2025-04-25T19:03:18.000+0000,null,2025-04-25T19:03:18.000+0000,null,{ },2022,OLIVIA,Albany,F,16
row-dbx8_dtbn-e5i4,00000000-0000-0000-CE32-1B645DB021DD,0,2025-04-25T19:03:18.000+0000,null,2025-04-25T19:03:18.000+0000,null,{ },2022,AMELIA,Albany,F,15
row-7bm2-ibrt_zigs,00000000-0000-0000-3BE2-1DBFC1A5B82A,0,2025-04-25T19:03:18.000+0000,null,2025-04-25T19:03:18.000+0000,null,{ },2022,AVERY,Albany,F,12
row-twbr~qzdf.jnfm,00000000-0000-0000-AAE1-DF2283C54B18,0,2025-04-25T19:03:18.000+0000,null,2025-04-25T19:03:18.000+0000,null,{ },2022,EMMA,Albany,F,11
row-hxpw_hv5d.a7xc,00000000-0000-0000-F7A1-34FD3134BFC6,0,2025-04-25T19:03:18.000+0000,null,2025-04-25T19:03:18.000+0000,null,{ },2022,CHARLOTTE,Albany,F,11
row-js7z_zax3_7jwq,00000000-0000-0000-C06B-5739366B5268,0,2025-04-25T19:03:18.000+0000,null,2025-04-25T19:03:18.000+0000,null,{ },2022,CHLOE,Albany,F,11
row-rp4d~vbca-bi5q,00000000-0000-0000-DA7E-7E1EC6BDA7E6,0,2025-04-25T19:03:18.000+0000,null,2025-04-25T19:03:18.000+0000,null,{ },2022,SOPHIA,Albany,F,8
row-eynv-94ax.jwpt,00000000-0000-0000-81AC-DF758F6AB3A3,0,2025-04-25T19:03:18.000+0000,null,2025-04-25T19:03:18.000+0000,null,{ },2022,CORA,Albany,F,8
row-6zwf.whw5.qpfr,00000000-0000-0000-87B7-D5BF3DC8B85F,0,2025-04-25T19:03:18.000+0000,null,2025-04-25T19:03:18.000+0000,null,{ },2022,MIA,Albany,F,7
row-vtb8_j2vg-fhgx,00000000-0000-0000-AADD-85B141D06470,0,2025-04-25T19:03:18.000+0000,null,2025-04-25T19:03:18.000+0000,null,{ },2022,LUNA,Albany,F,7


### Overwriting Table

In [0]:
(
    ny_health_data
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(NY_HEALTH_DATA)
)